# Lab 06 · Visualización de datos

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 6*

Trabaja sobre tu propia copia del notebook. Todo lo que escribas queda en ella.

In [ ]:
#@title De qué se trata este lab { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">De qué se trata este lab</div><strong>Preguntas que vamos a responder</strong>
<ul>
<li>Qué tiene que llevar un gráfico para que alguien lo entienda sin que tú estés al lado</li>
<li>Cuál de los cuatro gráficos básicos corresponde a cada pregunta</li>
<li>Cómo se muestra una serie de tiempo larga sin que se vea un borrón</li>
<li>Cuándo un gráfico interactivo aporta y cuándo estorba</li>
</ul>
<strong>Al terminar vas a poder</strong>
<ul>
<li>Armar un gráfico con matplotlib controlando cada parte</li>
<li>Elegir el tipo de gráfico a partir de la pregunta, no del gusto</li>
<li>Comparar grupos y distribuciones con seaborn en pocas líneas</li>
<li>Publicar un gráfico interactivo en un archivo que se manda por correo</li>
<li>Detectar un gráfico que engaña, incluso cuando lo hiciste tú</li>
</ul></div>"""))

In [ ]:
#@title Datos del curso { display-mode: "form" }
#@markdown Corre esta celda. Deja listos los archivos del Observatorio de Datos Energeticos.
import numpy as np, pandas as pd, os, json, sqlite3
if not os.path.exists("centrales.csv"):
    rng = np.random.default_rng(2026)
    centrales = pd.DataFrame([
     ("Central Rio Manso Alto","hidro","Biobio",420,2004),("Central Salto Verde","hidro","Los Lagos",310,1998),
     ("Central Aguas Claras","hidro","Biobio",180,2011),("Central Vega Azul","hidro","Los Lagos",95,2016),
     ("Central Tres Saltos","hidro","Biobio",260,1995),
     ("Parque Solar Pampa Alta","solar","Antofagasta",230,2019),("Parque Solar Llano Seco","solar","Atacama",180,2020),
     ("Parque Solar Sol Naciente","solar","Antofagasta",145,2021),("Parque Solar Quebrada Honda","solar","Atacama",95,2022),
     ("Parque Solar Altiplano","solar","Antofagasta",310,2023),
     ("Eolica Cerro Negro","eolica","Coquimbo",160,2017),("Eolica Punta Ventosa","eolica","Coquimbo",120,2018),
     ("Eolica Loma Fria","eolica","Valparaiso",85,2020),("Eolica Campo Abierto","eolica","Coquimbo",200,2021),
     ("Termoelectrica Bahia Norte","gas","Valparaiso",375,2008),("Termoelectrica Puerto Sur","gas","Biobio",290,2012),
     ("Termoelectrica Valle Central","gas","Metropolitana",210,2006),
     ("Carboelectrica Costa Brava","carbon","Biobio",480,2001),("Carboelectrica Roca Gris","carbon","Antofagasta",350,1999),
     ("Diesel Respaldo Cordillera","diesel","Metropolitana",45,2014),
    ], columns=["central","tecnologia","region","potencia_mw","anio_inicio"])
    fechas = pd.date_range("2024-01-01","2024-12-31",freq="D")
    perfil = np.array([0,0,0,0,0,0,.05,.18,.38,.58,.75,.87,.93,.9,.8,.63,.42,.2,.05,0,0,0,0,0])
    filas=[]
    for _,c in centrales.iterrows():
        p,t = c.potencia_mw, c.tecnologia
        for f in fechas:
            est = 1+0.25*np.cos(2*np.pi*(f.dayofyear-15)/365)
            if t=="solar": base = p*perfil*0.30*est*rng.uniform(.8,1.1)
            elif t=="eolica": base = p*0.36*rng.uniform(.15,1.6,24)
            elif t=="hidro": base = p*0.55*(2-est)*rng.uniform(.9,1.1,24)
            elif t=="gas": base = p*0.68*rng.uniform(.9,1.05,24)
            elif t=="carbon": base = p*0.65*rng.uniform(.95,1.02,24)
            else:
                base = np.zeros(24); base[18:23] = p*0.55*rng.uniform(.8,1,5)
            filas.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                       "central":c.central,"mwh":np.clip(base,0,p).round(2)}))
    centrales.to_csv("centrales.csv", index=False)
    pd.concat(filas, ignore_index=True).to_csv("generacion.csv", index=False)

    # Excel con dos hojas, la segunda con notas en texto libre
    with pd.ExcelWriter("centrales.xlsx") as w:
        centrales.to_excel(w, sheet_name="centrales", index=False)
        pd.DataFrame({"nota":["Potencias declaradas al 31 de diciembre de 2024",
                              "Las centrales de pasada se informan con su potencia maxima"]}
                     ).to_excel(w, sheet_name="notas", index=False)

    # Demanda por región, base de datos SQLite
    regs = ["Antofagasta","Atacama","Coquimbo","Valparaiso","Metropolitana","Biobio","Los Lagos"]
    pobl = [700000,320000,850000,1900000,8100000,1700000,900000]
    perfil_d = np.array([.72,.68,.66,.65,.66,.70,.78,.88,.95,.98,1.0,1.02,1.03,1.0,.97,.96,.97,1.0,1.06,1.10,1.08,.98,.88,.79])
    dem=[]
    for r,p in zip(regs,pobl):
        base_r = p/8000
        for f in fechas:
            inv = 1+0.18*np.cos(2*np.pi*(f.dayofyear-190)/365)
            finde = 0.92 if f.dayofweek>=5 else 1.0
            v = base_r*perfil_d*inv*finde*rng.uniform(.97,1.03,24)
            dem.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                     "region":r,"mwh":v.round(2)}))
    demanda = pd.concat(dem, ignore_index=True)
    demanda.to_csv("demanda.csv", index=False)
    con = sqlite3.connect("demanda.db")
    demanda.to_sql("demanda", con, index=False, if_exists="replace")
    pd.DataFrame({"region":regs,"poblacion":pobl}).to_sql("regiones", con, index=False, if_exists="replace")
    con.close()

    # Precios de nudo de enero, como los entregaría una API REST
    ene = demanda[demanda["fecha"].str.startswith("2024-01")]
    pr = ene.assign(precio_usd_mwh=(40 + ene["mwh"]/ene["mwh"].max()*110
                                    + rng.normal(0,6,len(ene))).clip(40,180).round(2))
    with open("precios_nudo.json","w") as f:
        json.dump({"metadata":{"fuente":"Observatorio de Datos Energeticos",
                               "fecha_consulta":"2024-02-01","unidad":"USD por MWh"},
                   "datos": pr[["fecha","hora","region","precio_usd_mwh"]].to_dict("records")},
                  f)

    # ------------------------------------------------ bitacora de mantenimiento
    # Seiscientos eventos del ano, con dos patrones plantados a proposito que el
    # lab va a tener que encontrar y medir.
    rngm = np.random.default_rng(404)
    TIPOS = ["preventivo","correctivo","falla_electrica","falla_mecanica",
             "evento_climatico","inspeccion"]
    PESOS = {"hidro":[.34,.14,.12,.16,.02,.22], "solar":[.38,.12,.16,.08,.02,.24],
             "eolica":[.26,.12,.12,.20,.08,.22], "gas":[.30,.18,.14,.18,.01,.19],
             "carbon":[.28,.18,.14,.20,.01,.19], "diesel":[.14,.52,.10,.12,.01,.11]}
    DURA = {"preventivo":(4,24), "correctivo":(6,72), "falla_electrica":(2,48),
            "falla_mecanica":(8,96), "evento_climatico":(3,36), "inspeccion":(1,8)}
    n_dias = len(fechas)
    invierno = np.isin(fechas.month.to_numpy(), [5,6,7,8])
    peso_clima = np.where(invierno, 4.0, 1.0); peso_clima /= peso_clima.sum()

    ev = []
    for _,c in centrales.iterrows():
        p = np.array(PESOS[c.tecnologia]); p = p/p.sum()
        n = 30 if c.tecnologia=="diesel" else 22
        for tipo, d in zip(rngm.choice(TIPOS, size=n, p=p), rngm.integers(0, n_dias, size=n)):
            ev.append([c.central, int(d), str(tipo)])
        n_cl = 12 if c.tecnologia=="eolica" else 1
        for d in rngm.choice(n_dias, size=n_cl, replace=False, p=peso_clima):
            ev.append([c.central, int(d), "evento_climatico"])

    # Patron 1, el 70 por ciento de las fallas electricas arrastra un correctivo
    # en la misma central dentro de los tres dias siguientes.
    elec = [i for i,e in enumerate(ev) if e[2]=="falla_electrica"]
    for i in sorted(rngm.choice(elec, size=int(round(len(elec)*0.70)), replace=False)):
        ev.append([ev[i][0], min(ev[i][1] + int(rngm.integers(0,4)), n_dias-1), "correctivo"])
    # Patron 2, la mitad de los eventos climaticos trae una falla mecanica el mismo dia.
    clim = [i for i,e in enumerate(ev) if e[2]=="evento_climatico"]
    for i in sorted(rngm.choice(clim, size=int(round(len(clim)*0.50)), replace=False)):
        ev.append([ev[i][0], ev[i][1], "falla_mecanica"])

    nombres = centrales["central"].to_numpy()
    while len(ev) < 600:
        ev.append([str(nombres[int(rngm.integers(0,len(nombres)))]),
                   int(rngm.integers(0,n_dias)),
                   "preventivo" if rngm.random()<0.6 else "inspeccion"])
    ev = ev[:600]

    mant = pd.DataFrame([{"central":c, "fecha":fechas[d].strftime("%Y-%m-%d"),
                          "tipo_evento":t,
                          "duracion_horas":int(rngm.integers(DURA[t][0], DURA[t][1]+1))}
                         for c,d,t in ev])

    # ------------------------------------------- texto libre de cada evento
    # Una observación escrita como la escribiría el turno, con vocabulario
    # propio de cada tipo. El Módulo 5 busca temas ahí adentro.
    VOCAB = {
     "falla_electrica": (["el transformador de poder","el interruptor principal",
        "la barra de media tensión","el relé de protección","el aislador de línea"],
        ["sobretensión sostenida","un cortocircuito monofásico","corriente de fuga elevada",
         "el disparo de la protección diferencial"],
        ["se aísla el circuito y se normaliza la tensión","se reemplaza el relé y se recalibra",
         "se reconecta el interruptor tras verificar la aislación"]),
     "falla_mecanica": (["el rodamiento del eje","la caja multiplicadora","el acoplamiento",
        "el sello del descanso","la bomba de lubricación"],
        ["vibración fuera de norma","temperatura elevada en el descanso","ruido anormal",
         "pérdida de aceite"],
        ["se reemplaza el rodamiento y se alinea el eje","se rellena y se purga el circuito de aceite",
         "se ajusta el acoplamiento y se mide la vibración"]),
     "evento_climatico": (["la línea de evacuación","el patio de alta tensión",
        "el camino de acceso","la estructura de la torre","el pararrayos del patio"],
        ["viento sobre lo previsto","una descarga atmosférica cercana","acumulación de nieve",
         "lluvia intensa con anegamiento"],
        ["se inspecciona la estructura y se despeja la faja","se repone el servicio al amainar",
         "se drena el sector y se revisa la puesta a tierra"]),
     "preventivo": (["el sistema de refrigeración","los filtros de aire","el tablero de control",
        "las conexiones de fuerza","el grupo hidráulico"],
        ["la mantención programada","el cambio de filtros","el ajuste de rutina",
         "la lubricación periódica"],
        ["se cambian filtros y se registra la lectura","se reaprietan las conexiones y se sella",
         "se completa la pauta sin observaciones"]),
     "correctivo": (["el equipo afectado","la unidad detenida","el componente dañado",
        "la sección fuera de servicio","el módulo de potencia"],
        ["la reparación de la falla del turno anterior","el levantamiento de la indisponibilidad",
         "la orden de trabajo pendiente","la intervención de emergencia"],
        ["se repara y se devuelve a servicio","se reemplaza la pieza y se prueba en vacío",
         "se normaliza y se informa al despacho"]),
     "inspeccion": (["el conjunto de medida","la señalética del área","los niveles de aceite",
        "el estado de los accesos","el registro de alarmas"],
        ["la ronda de rutina","la verificación visual","la lectura de instrumentos",
         "el chequeo de seguridad"],
        ["se deja constancia sin hallazgos","se anota una observación menor",
         "se programa revisión de detalle"]),
    }
    PLANT = ["Se registra {s} sobre {c}.",
             "Se detecta {s} en {c}, {a}.",
             "El operador reporta {s}, se revisa {c} y {a}.",
             "Evento por {s} en {c}, {a}."]
    def _obs(t):
        c, s, a = (VOCAB[t][k][int(rngm.integers(0, len(VOCAB[t][k])))] for k in (0, 1, 2))
        return PLANT[int(rngm.integers(0, len(PLANT)))].format(s=s, c=c, a=a)
    mant["observacion"] = [_obs(t) for t in mant["tipo_evento"]]
    mant.sort_values(["fecha","central"], kind="stable").reset_index(drop=True) \
        .to_csv("mantenimiento.csv", index=False)
print("Datos listos")


## 1. Anatomía de un gráfico

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

gen = pd.read_csv("generacion.csv", parse_dates=["fecha"])
centrales = pd.read_csv("centrales.csv")
demanda = pd.read_csv("demanda.csv", parse_dates=["fecha"])
gen = gen.merge(centrales, on="central")

print(gen.shape, demanda.shape)

In [ ]:
# El gráfico más corto posible. Sale, y no dice nada.
met = demanda[demanda["region"] == "Metropolitana"]
semana = met[met["fecha"].between("2024-06-03", "2024-06-09")]

fig, ax = plt.subplots()
ax.plot(semana["mwh"].values)
plt.show()

In [ ]:
# El mismo dato, con lo mínimo que hace falta para entenderlo.
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(semana["fecha"] + pd.to_timedelta(semana["hora"], unit="h"), semana["mwh"])
ax.set_title("Demanda de la Región Metropolitana, primera semana de junio de 2024")
ax.set_xlabel("")
ax.set_ylabel("MWh por hora")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
#@title Lo mínimo que lleva un gráfico { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Lo mínimo que lleva un gráfico</div><p>Un <strong>título</strong> que diga qué se está mirando, con el periodo adentro. Si el título no alcanza para entender el gráfico, el título está mal.</p>
<p>Los <strong>ejes rotulados con su unidad</strong>. MWh por hora, no solo MWh. El eje que se explica solo, como una fecha, no necesita rótulo.</p>
<p>Una <strong>escala honesta</strong>. Si el eje no parte en cero, hay que poder justificarlo.</p>
<p>Y nada más. Cada elemento que agregas tiene que ganarse el lugar.</p></div>"""))

In [ ]:
# fig y ax son dos cosas distintas y conviene tenerlo claro.
# fig es la hoja, ax es el dibujo. En una hoja puede haber varios dibujos.
fig, ejes = plt.subplots(1, 2, figsize=(10, 3))
ejes[0].plot(semana["mwh"].values)
ejes[0].set_title("un dibujo")
ejes[1].bar(["a", "b", "c"], [3, 1, 2])
ejes[1].set_title("y otro en la misma hoja")
fig.suptitle("Una hoja, dos dibujos", y=1.04)
plt.show()

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>El mismo gráfico, otra región</strong>
<p>Cambia la Metropolitana por Valparaiso y la semana de junio por una de enero. Fíjate en que el título tiene que cambiar solo si lo escribes con una variable adentro.</p></div>"""))

In [ ]:
# Tu turno
# Cambia la región y las fechas, y arma el título con una f-string.
region, desde, hasta = "Metropolitana", "2024-06-03", "2024-06-09"
trozo = demanda[(demanda["region"] == region) & (demanda["fecha"].between(desde, hasta))]

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(trozo["fecha"] + pd.to_timedelta(trozo["hora"], unit="h"), trozo["mwh"])
ax.set_title(f"Demanda de {region}, del {desde} al {hasta}")
ax.set_ylabel("MWh por hora")
plt.show()

## 2. Los cuatro gráficos básicos

In [ ]:
# Línea, cuando el eje horizontal tiene orden. Casi siempre, tiempo.
diaria = gen.groupby("fecha")["mwh"].sum() / 1000

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(diaria.index, diaria.values, linewidth=0.9)
ax.set_title("Generación diaria del sistema, 2024")
ax.set_ylabel("GWh por día")
plt.show()

In [ ]:
# Barras, cuando se comparan categorías. Horizontales si los nombres son largos.
por_tec = gen.groupby("tecnologia")["mwh"].sum().sort_values() / 1000

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(por_tec.index, por_tec.values, color="#4C78A8")
ax.set_title("Generación del año por tecnología")
ax.set_xlabel("GWh")
plt.show()

In [ ]:
# Dispersión, cuando la pregunta es si dos números se mueven juntos.
resumen = gen.groupby(["central", "tecnologia", "potencia_mw"])["mwh"].sum().reset_index()
resumen["gwh"] = resumen["mwh"] / 1000

fig, ax = plt.subplots(figsize=(7, 4.5))
for tec, grupo in resumen.groupby("tecnologia"):
    ax.scatter(grupo["potencia_mw"], grupo["gwh"], label=tec, s=60)
ax.set_title("Potencia instalada contra generación del año")
ax.set_xlabel("potencia instalada (MW)")
ax.set_ylabel("generación (GWh)")
ax.legend(fontsize=9)
plt.show()

In [ ]:
# Histograma, cuando la pregunta es cómo se reparte una sola variable.
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(met["mwh"], bins=40, color="#4C78A8")
ax.set_title("Cómo se reparten las 8.784 horas de demanda de la Metropolitana")
ax.set_xlabel("MWh por hora")
ax.set_ylabel("cuántas horas")
plt.show()

In [ ]:
# Los cuatro juntos, para tenerlos a la vista.
fig, ejes = plt.subplots(2, 2, figsize=(11, 6))
ejes[0, 0].plot(diaria.index, diaria.values, linewidth=0.7)
ejes[0, 0].set_title("línea, el tiempo manda")
ejes[0, 1].barh(por_tec.index, por_tec.values, color="#4C78A8")
ejes[0, 1].set_title("barras, comparar categorías")
ejes[1, 0].scatter(resumen["potencia_mw"], resumen["gwh"], s=30)
ejes[1, 0].set_title("dispersión, dos números")
ejes[1, 1].hist(met["mwh"], bins=30, color="#4C78A8")
ejes[1, 1].set_title("histograma, una variable")
fig.tight_layout()
plt.show()

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Barras por región</strong>
<p>Arma el gráfico de barras horizontales de la demanda total del año por región, ordenado de menor a mayor. Es el mismo patrón del gráfico por tecnología. Fíjate en que ordenar antes de dibujar cambia por completo cómo se lee.</p></div>"""))

In [ ]:
# Tu turno
# Cambia gen por demanda y tecnología por región.
por_region = gen.groupby("tecnologia")["mwh"].sum().sort_values() / 1000

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(por_region.index, por_region.values, color="#72B7B2")
ax.set_title("Cambia esto por la demanda por región")
plt.show()

## 3. Comparar grupos con seaborn

In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid")

# Un boxplot compara la distribución de varios grupos de una vez.
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=demanda, x="region", y="mwh", ax=ax)
ax.set_title("Demanda horaria por región, todo el año")
ax.set_ylabel("MWh por hora")
ax.set_xlabel("")
plt.show()

In [ ]:
# La Metropolitana aplasta a las demás. Sin ella se ven las otras seis.
sin_rm = demanda[demanda["region"] != "Metropolitana"]

fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=sin_rm, x="region", y="mwh", ax=ax)
ax.set_title("Las mismas seis regiones, sin la Metropolitana en la escala")
ax.set_ylabel("MWh por hora")
ax.set_xlabel("")
plt.show()

In [ ]:
# Un mapa de calor muestra dos categorías y un número, todo de una vez.
perfil = gen.pivot_table(index="tecnologia", columns="hora", values="mwh", aggfunc="mean")

fig, ax = plt.subplots(figsize=(11, 3))
sns.heatmap(perfil, cmap="YlOrRd", ax=ax, cbar_kws={"label": "MWh promedio"})
ax.set_title("El día promedio de cada tecnología")
ax.set_xlabel("hora")
ax.set_ylabel("")
plt.show()

In [ ]:
# El mismo mapa, con cada fila normalizada por su propio máximo.
# Ahora se compara la forma del día, no el tamaño de la tecnología.
perfil_n = perfil.div(perfil.max(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(11, 3))
sns.heatmap(perfil_n, cmap="YlOrRd", ax=ax, cbar_kws={"label": "proporción del máximo"})
ax.set_title("La forma del día de cada tecnología, sin el tamaño")
ax.set_xlabel("hora")
ax.set_ylabel("")
plt.show()

In [ ]:
#@title Dos mapas de calor, dos preguntas { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Dos mapas de calor, dos preguntas</div><p>El primero responde cuánto genera cada tecnología a cada hora. La fila de la hidro es la más caliente porque es la que más genera, y las otras casi no se distinguen entre sí.</p>
<p>El segundo responde a qué hora genera cada tecnología. Todas las filas usan el mismo rango, así que se ve la forma. La solar aparece como una banda de mediodía y el diésel como una banda de noche.</p>
<p>Es el mismo dato y la misma función. Lo que cambió es una línea de normalización, igual que en el Módulo 5.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>El mapa de la demanda</strong>
<p>Arma el mismo mapa de calor normalizado, pero con la demanda por región en vez de la generación por tecnología. Vas a ver si las siete regiones tienen la misma forma del día.</p></div>"""))

In [ ]:
# Tu turno
# Cambia gen por demanda y tecnología por región.
perfil_d = gen.pivot_table(index="tecnologia", columns="hora", values="mwh", aggfunc="mean")

fig, ax = plt.subplots(figsize=(11, 3))
sns.heatmap(perfil_d.div(perfil_d.max(axis=1), axis=0), cmap="YlOrRd", ax=ax)
ax.set_title("Cambia esto por la demanda por región")
plt.show()

## 4. Series de tiempo

In [ ]:
# 8.784 puntos en una línea es un borrón. Esto es lo que NO hay que entregar.
met_h = met.copy()
met_h["instante"] = met_h["fecha"] + pd.to_timedelta(met_h["hora"], unit="h")
met_h = met_h.set_index("instante").sort_index()

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(met_h.index, met_h["mwh"], linewidth=0.4)
ax.set_title("Las 8.784 horas del año, dibujadas todas")
plt.show()

In [ ]:
# resample agrupa por tiempo. D es día, W semana, ME fin de mes.
por_dia = met_h["mwh"].resample("D").mean()

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(por_dia.index, por_dia.values, linewidth=1)
ax.set_title("El promedio de cada día, que ya se lee")
ax.set_ylabel("MWh por hora")
plt.show()

In [ ]:
# Una media móvil suaviza sin cambiar la escala del eje.
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(por_dia.index, por_dia.values, linewidth=0.7, alpha=0.45, label="día a día")
ax.plot(por_dia.index, por_dia.rolling(14, center=True).mean(),
        linewidth=2, color="#E45756", label="media móvil de 14 días")
ax.set_title("El dato crudo y su tendencia, en el mismo gráfico")
ax.set_ylabel("MWh por hora")
ax.legend()
plt.show()

In [ ]:
# Una banda entre el mínimo y el máximo del día dice más que el promedio solo.
minimo = met_h["mwh"].resample("D").min()
maximo = met_h["mwh"].resample("D").max()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.fill_between(por_dia.index, minimo, maximo, alpha=0.25, color="#4C78A8",
                label="entre el mínimo y el máximo del día")
ax.plot(por_dia.index, por_dia.values, linewidth=1, color="#1F4E79", label="promedio del día")
ax.set_title("Cada día tiene un rango, no un número")
ax.set_ylabel("MWh por hora")
ax.legend(fontsize=9)
plt.show()

In [ ]:
# Y el perfil del día promedio, verano contra invierno.
met_h["mes"] = met_h.index.month
verano = met_h[met_h["mes"].isin([12, 1, 2])].groupby("hora")["mwh"].mean()
invierno = met_h[met_h["mes"].isin([6, 7, 8])].groupby("hora")["mwh"].mean()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(verano.index, verano.values, marker="o", label="verano")
ax.plot(invierno.index, invierno.values, marker="o", label="invierno")
ax.set_title("El día promedio de la Metropolitana, por estación")
ax.set_xlabel("hora del día")
ax.set_ylabel("MWh por hora")
ax.legend()
plt.show()

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Otra ventana</strong>
<p>Cambia el <code>resample("D")</code> por <code>resample("W")</code> y la media móvil de 14 días por una de 4 semanas. Mira si la historia que cuenta el gráfico cambia o solo cambia el ruido.</p></div>"""))

In [ ]:
# Tu turno
# Cambia la D por una W y ajusta la ventana de la media móvil.
serie = met_h["mwh"].resample("D").mean()

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(serie.index, serie.values, linewidth=0.8)
ax.set_title("Cambia la frecuencia del resample")
plt.show()

In [ ]:
#@title Hasta acá llega la primera clase { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Hasta acá llega la primera clase</div><p>Con lo de hoy ya puedes armar cualquiera de los gráficos que aparecen en un informe del sector. Los cuatro básicos, la comparación entre grupos y la serie de tiempo bien resumida.</p>
<p>La próxima clase, que es corta, cierra con tres cosas. Las distribuciones, los gráficos interactivos y, sobre todo, cómo se revisa un gráfico antes de mandarlo.</p>
<p><strong>Cuando vuelvas, ejecuta el notebook desde arriba.</strong> Entorno de ejecución, Ejecutar todo. Colab no guarda el estado entre sesiones.</p></div>"""))

## 5. Distribuciones

In [ ]:
# El promedio solo esconde la forma. Cuatro regiones, cuatro histogramas.
cuatro = ["Metropolitana", "Biobio", "Valparaiso", "Atacama"]

fig, ejes = plt.subplots(2, 2, figsize=(9, 5))
for eje, region in zip(ejes.ravel(), cuatro):
    serie = demanda[demanda["region"] == region]["mwh"]
    eje.hist(serie, bins=30, color="#4C78A8")
    eje.set_title(f"{region}\npromedio {serie.mean():.0f}")
    eje.set_xlabel("MWh por hora")
fig.tight_layout()
plt.show()

In [ ]:
# El violín muestra la forma y la caja al mismo tiempo.
tres = demanda[demanda["region"].isin(["Metropolitana", "Biobio", "Los Lagos"])]

fig, ax = plt.subplots(figsize=(8, 4))
sns.violinplot(data=tres, x="region", y="mwh", ax=ax, inner="quartile")
ax.set_title("La forma completa, no solo la caja")
ax.set_ylabel("MWh por hora")
ax.set_xlabel("")
plt.show()

In [ ]:
# La trampa clásica. El promedio no dice nada sobre la variabilidad.
diario = gen.groupby(["tecnologia", "fecha"])["mwh"].sum().unstack(0)
resumen = pd.DataFrame({"promedio": diario.mean().round(1), "desviacion": diario.std().round(1)})
resumen["cv_pct"] = (100 * resumen["desviacion"] / resumen["promedio"]).round(1)

print(resumen.sort_values("cv_pct", ascending=False).to_string())

In [ ]:
# El carbón entrega siete veces más energía que la solar y varía sesenta veces
# menos. Con la escala de cada uno, para que se vea la forma y no el tamaño.
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.2))
for eje, tec, color in [(ejes[0], "carbon", "#7F5F52"), (ejes[1], "solar", "#F58518")]:
    serie = diario[tec]
    eje.hist(serie, bins=30, color=color)
    eje.set_title(f"{tec}\npromedio {serie.mean():.0f}, desviación {serie.std():.0f}")
    eje.set_xlabel("MWh por día")
fig.tight_layout()
plt.show()

In [ ]:
#@title Ojo con la eólica de estos datos { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Ojo con la eólica de estos datos</div><p>En la tabla de arriba la eólica sale como la tecnología más estable del parque, con un coeficiente de variación de 5,1 por ciento contra 18,1 de la solar. <strong>En el sistema real es al revés.</strong></p>
<p>Es un artefacto de cómo se generaron estos datos. La eólica se construyó con ruido independiente hora a hora, y al sumar las veinticuatro horas de cuatro parques ese ruido se promedia y desaparece. En el sistema real el viento tiene rachas que duran días, así que la variabilidad no se cancela.</p>
<p>La solar, en cambio, sí varía de verdad en estos datos, porque lleva el ciclo estacional adentro.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>El factor de planta</strong>
<p>Calcula el factor de planta de cada central, que es su generación del año dividida por su potencia por las horas del año, y dibuja su distribución con un histograma. Vas a ver que no es una sola nube.</p></div>"""))

In [ ]:
# Tu turno
# Divide la generación anual de cada central por potencia_mw por 8784.
anual = gen.groupby(["central", "potencia_mw"])["mwh"].sum().reset_index()

print(anual.head(3).to_string(index=False))

## 6. Gráficos interactivos

In [ ]:
import plotly.express as px

# Plotly arma el gráfico desde el DataFrame, sin fig ni ax.
diaria_region = (demanda.groupby(["fecha", "region"])["mwh"].mean().reset_index())

fig = px.line(diaria_region, x="fecha", y="mwh", color="region",
              title="Demanda horaria promedio por día y región",
              labels={"mwh": "MWh por hora", "fecha": ""})
fig.show()

In [ ]:
# Con una dimensión más, el color y el tamaño cuentan otra cosa.
resumen = gen.groupby(["central", "tecnologia", "potencia_mw"])["mwh"].sum().reset_index()
resumen["gwh"] = (resumen["mwh"] / 1000).round(1)

fig = px.scatter(resumen, x="potencia_mw", y="gwh", color="tecnologia",
                 size="gwh", hover_name="central",
                 title="Potencia instalada contra generación, por central",
                 labels={"potencia_mw": "potencia instalada (MW)", "gwh": "generación (GWh)"})
fig.show()

In [ ]:
# Un gráfico interactivo se guarda como un archivo que se abre en cualquier navegador.
fig.write_html("centrales_interactivo.html")

import os
print("archivo escrito,", round(os.path.getsize("centrales_interactivo.html") / 1024), "KB")

In [ ]:
#@title Cuándo el interactivo aporta y cuándo estorba { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Cuándo el interactivo aporta y cuándo estorba</div><p><strong>Aporta</strong> cuando hay muchas series y quien mira quiere aislar una. Cuando el nombre de cada punto importa y no cabe en la etiqueta. Y cuando el archivo se va a mandar por correo para que otro lo explore.</p>
<p><strong>Estorba</strong> cuando el gráfico va a terminar impreso o pegado en una presentación, porque ahí se pierde todo lo interactivo y queda un gráfico peor maquetado. También cuando hay un solo mensaje que dar, porque la interacción invita a buscar otro.</p>
<p>La regla corta. Si el gráfico responde una pregunta, estático. Si sirve para que otro explore, interactivo.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Barras apiladas por mes</strong>
<p>Arma con <code>px.bar</code> la generación mensual por tecnología, apilada. Pista, necesitas una columna de mes y <code>color="tecnologia"</code>. Fíjate en que el orden de las categorías lo decide plotly y se puede fijar.</p></div>"""))

In [ ]:
# Tu turno
# Agrupa por mes y tecnología y pásaselo a px.bar con color y barmode.
gen["mes"] = gen["fecha"].dt.month
mensual = (gen.groupby(["mes", "tecnologia"])["mwh"].sum() / 1000).round(1).reset_index()

print(mensual.head(4).to_string(index=False))

## 7. Antes y después

In [ ]:
# Un gráfico real, hecho rápido. Todo lo que tiene está mal.
por_central = gen.groupby("central")["mwh"].sum() / 1000

fig, ax = plt.subplots()
ax.bar(por_central.index, por_central.values)
plt.show()

In [ ]:
# El mismo dato, arreglado. Cuatro cambios, ni uno de ellos es sofisticado.
orden = por_central.sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(orden.index, orden.values, color="#4C78A8")
ax.set_title("Generación del año por central, 2024")
ax.set_xlabel("GWh")
ax.bar_label(ax.containers[0], fmt="%.0f", padding=3, fontsize=9)
ax.grid(axis="x", visible=False)
ax.set_xlim(0, orden.max() * 1.12)
plt.show()

In [ ]:
#@title Los cuatro arreglos { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Los cuatro arreglos</div><ul>
<li><strong>Barras horizontales.</strong> Los nombres largos se leen, no hay que girar la cabeza.</li>
<li><strong>Ordenado.</strong> Un gráfico de barras sin ordenar obliga a buscar. Ordenado, la respuesta salta.</li>
<li><strong>Título con el periodo.</strong> Sin eso, el gráfico no se puede archivar ni citar.</li>
<li><strong>El valor al lado de la barra.</strong> Cuando son pocas categorías, el número exacto evita que el lector estime mal.</li>
</ul>
<p>Ninguno de los cuatro es una técnica. Son decisiones de quien dibuja.</p></div>"""))

In [ ]:
# La escala que engaña. El mismo dato, dos historias.
tres_meses = por_central.sort_values(ascending=False).head(3)

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5))
ejes[0].bar(range(3), tres_meses.values, color="#E45756")
ejes[0].set_ylim(tres_meses.min() * 0.985, tres_meses.max() * 1.005)
ejes[0].set_title("Eje cortado, parece el doble")
ejes[1].bar(range(3), tres_meses.values, color="#4C78A8")
ejes[1].set_ylim(0, tres_meses.max() * 1.1)
ejes[1].set_title("Eje desde cero, la diferencia real")
for eje in ejes:
    eje.set_xticks(range(3))
    eje.set_xticklabels([c.split()[-1] for c in tres_meses.index], fontsize=8)
    eje.set_ylabel("GWh")
fig.tight_layout()
plt.show()

In [ ]:
# Y el color. Esta paleta la distingue cualquiera, incluido quien no ve rojo y verde.
fig, ax = plt.subplots(figsize=(9, 3.5))
colores = sns.color_palette("colorblind", n_colors=6)
for (tec, grupo), color in zip(gen.groupby("tecnologia"), colores):
    serie = grupo.groupby("hora")["mwh"].mean()
    ax.plot(serie.index, serie.values, label=tec, color=color, linewidth=2)
ax.set_title("El día promedio de cada tecnología, con paleta segura")
ax.set_xlabel("hora del día")
ax.set_ylabel("MWh promedio")
ax.legend(fontsize=9)
plt.show()

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Revisa un gráfico tuyo</strong>
<p>Toma cualquiera de los gráficos que hiciste hoy y pásale la lista de los cuatro arreglos. Título con periodo, ejes con unidad, orden si son categorías, y escala desde cero salvo que puedas justificar lo contrario. Casi seguro le falta alguno.</p></div>"""))

In [ ]:
# Tu turno
# Pega acá el gráfico que quieras revisar y arréglalo.
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(por_dia.index, por_dia.values)
plt.show()

In [ ]:
#@title Puntos clave { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Puntos clave</div><ul>
<li>Un gráfico sin título con periodo y sin unidad en los ejes no se puede archivar ni citar</li>
<li>El tipo de gráfico sale de la pregunta, línea para el tiempo, barras para categorías, dispersión para dos números e histograma para uno</li>
<li>Una serie de 8.784 puntos se resume con <code>resample</code> y una media móvil, no se dibuja entera</li>
<li>Normalizar cada fila de un mapa de calor cambia la pregunta que responde</li>
<li>El interactivo sirve para explorar, el estático para responder</li>
<li>Cortar el eje vertical en un gráfico de barras exagera la diferencia, y hay que poder justificarlo</li>
</ul>
<p>En el caso integrador vas a usar los seis módulos sobre datos de un año que todavía no viste.</p></div>"""))